# XAI CODE DEMO

[![Open In Collab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AIPI-590-XAI/Duke-AI-XAI/blob/dev/interpretable-ml-example-notebooks/generalized-models-interpretability.ipynb)

# Generalized Models

* Generalized Linear Models (GLMs)
* Generalized Additive Models (GAMs)

In [1]:
!pip install pygam --quiet

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from statsmodels.genmod.generalized_linear_model import GLM
from statsmodels.genmod.families import Gamma
from statsmodels.genmod.families.links import Log
from statsmodels.tools import add_constant
from pygam import LinearGAM, GammaGAM, s
import seaborn as sns

## 1. Dataset loading and cleaning.

#### Dataset

We will be using the [Telco Customer Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn/code)

In [3]:
import kagglehub
import pandas as pd
import os

# Download latest version
path = kagglehub.dataset_download("blastchar/telco-customer-churn")

print("Path to dataset files:", path, "\n")

# Read the CSV
df = pd.read_csv(os.path.join(path, "WA_Fn-UseC_-Telco-Customer-Churn.csv"))

# Preview
display(df.head())

Using Colab cache for faster access to the 'telco-customer-churn' dataset.
Path to dataset files: /kaggle/input/telco-customer-churn 



,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [5]:
for column in df.columns:
    print(f"Column: {column} ({df[column].nunique()})")
    print(df[column].unique())
    print("\n")

Column: customerID (7043)
['7590-VHVEG' '5575-GNVDE' '3668-QPYBK' ... '4801-JZAZL' '8361-LTMKD'
 '3186-AJIEK']


Column: gender (2)
['Female' 'Male']


Column: SeniorCitizen (2)
[0 1]


Column: Partner (2)
['Yes' 'No']


Column: Dependents (2)
['No' 'Yes']


Column: tenure (73)
[ 1 34  2 45  8 22 10 28 62 13 16 58 49 25 69 52 71 21 12 30 47 72 17 27
  5 46 11 70 63 43 15 60 18 66  9  3 31 50 64 56  7 42 35 48 29 65 38 68
 32 55 37 36 41  6  4 33 67 23 57 61 14 20 53 40 59 24 44 19 54 51 26  0
 39]


Column: PhoneService (2)
['No' 'Yes']


Column: MultipleLines (3)
['No phone service' 'No' 'Yes']


Column: InternetService (3)
['DSL' 'Fiber optic' 'No']


Column: OnlineSecurity (3)
['No' 'Yes' 'No internet service']


Column: OnlineBackup (3)
['Yes' 'No' 'No internet service']


Column: DeviceProtection (3)
['No' 'Yes' 'No internet service']


Column: TechSupport (3)
['No' 'Yes' 'No internet service']


Column: StreamingTV (3)
['No' 'Yes' 'No internet service']


Column: StreamingMovies 

Convert to numeric

In [6]:
for feat in ['MonthlyCharges', 'TotalCharges']:
  # Convert the feature to numeric, handling potential non-numeric entries (empty strings) as NaN.
  df[feat] = pd.to_numeric(df[feat], errors='coerce')

  # Fill any NaN values that resulted from the conversion (e.g., empty strings) with 0.
  df[feat] = df[feat].fillna(0)

Convert to binary

In [7]:
# Convert feat to object type to treat it as a categorical feature.
for feat in ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']:
  df[feat] = df[feat].map({'Yes': 1, 'No': 0})

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   int64  
 4   Dependents        7043 non-null   int64  
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   int64  
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   int64  


## 2. EDA relevant to GAM assumptions.

In [ ]:
# Load the Telco dataset
# df = pd.read_csv(path)
# df.head(3)
# X, y = diabetes.data, diabetes.target

# The target variable needs to be positive for Gamma GLM
y = y - y.min() + 1  # Shift and scale to ensure all values are positive

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Load the Diabetes dataset
diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target

# The target variable needs to be positive for Gamma GLM
y = y - y.min() + 1  # Shift and scale to ensure all values are positive

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Generalized Additive Model (GAM)

* GAMs assume that the outcome can be modeled by a sum of arbitrary functions of each feature
* GAMs use B splines, which allow us to automatically model non-linear relationships
* GAM is still a sum of feature effects, but it gives the option to allow nonlinear relationships between some features and the output

#### Terminology
* LinearGAM: uses a Normal (Gaussian) error distribution
* s(*i*): Represents a spline term for the *i*th feature, allowing for flexible, non-linear relationships between the feature and the response variable

We are using the pygam library. Documentation [here](https://pygam.readthedocs.io/en/latest/notebooks/tour_of_pygam.html).

* The gridsearch method performs a grid search over a pre-defined range of hyperparameters to find the best set of hyperparameters (smoothing parameters for each spline term). The method evaluates different combinations of hyperparameters using cross-validation on the training data and selects the combination that minimizes the cross-validation error.

In [ ]:
# Initilize GAM
gam = LinearGAM(s(0) + s(1) + s(2) + s(3) + s(4) + s(5) + s(6) + s(7) + s(8) + s(9))

# Find best smoothing parameters for each spline term
gam.gridsearch(X_train_scaled, y_train)

# Fit the model
gam.fit(X_train_scaled, y_train)

# Make predictions
y_pred_gam = gam.predict(X_test_scaled)

# Calculate MSE and R^2
mse_gam = mean_squared_error(y_test, y_pred_gam)
r2_gam = r2_score(y_test, y_pred_gam)

print(f"GAM MSE: {mse_gam:.4f}, R^2: {r2_gam:.4f}")

gam.summary()

### GAM - Partial Dependence Plots
Plots the partial dependence of each feature, showing the effect of each feature on the predicted outcome while holding other features constant.

In [ ]:
# Visualize GAM
plt.figure(figsize=(20, 15))
for i, term in enumerate(gam.terms):
    if term.isintercept:
        continue
    plt.subplot(4, 3, i+1)
    XX = gam.generate_X_grid(term=i)
    plt.plot(XX[:, term.feature], gam.partial_dependence(term=i, X=XX))
    plt.title(diabetes.feature_names[i])
    plt.ylabel('Partial dependence')
plt.tight_layout()
plt.show()